# Zugabe: Eine kleine Textanalyse

Notebooks können nicht nur Zahlen. In diesem Notebook zählst du Wörter in Grimms Märchen, zeichnest die häufigsten und vergleichst zwei Märchen miteinander.

**Die Texte:** Zehn Märchen aus der Erstausgabe der *Kinder- und Hausmärchen* (1812/15) der Brüder Grimm, als einzelne Textdateien im Ordner `daten/maerchen/`. Die Schreibweise ist die von 1812, also "Rothkäppchen" und "daß". Quelle ist die Digitale Bibliothek des [TextGrid-Repositorys](https://textgridrep.de/browse/nxsj.0), Lizenz CC BY 3.0 DE. Details stehen in `daten/maerchen/QUELLE.md`.

## Welche Märchen gibt es?

In [ ]:
from pathlib import Path

ordner = Path("../daten/maerchen")

for datei in sorted(ordner.glob("*.txt")):
    print(datei.name)

## Einen Text einlesen

Eine Textdatei ist in Python eine einzige lange Zeichenkette. Wir lesen Hänsel und Gretel ein und schauen uns den Anfang an.

**Aufgabe:** Ändere den Dateinamen in ein anderes Märchen aus der Liste oben.

In [ ]:
datei = ordner / "15_haensel_und_gretel.txt"

text = datei.read_text(encoding="utf-8")

print(f"{len(text)} Zeichen")
print()
print(text[:500])

## In Wörter zerlegen

Um Wörter zu zählen, zerlegen wir den Text. `re.findall(r"[^\W\d_]+",...)` findet alle zusammenhängenden Buchstabenfolgen, also die Wörter ohne Satzzeichen und Ziffern. Das kryptische Muster ist ein regulärer Ausdruck, man muss ihn nicht verstehen, um ihn zu benutzen. `lower()` macht alles klein, damit "Hänsel" und "hänsel" als ein Wort zählen.

In [ ]:
import re

woerter = re.findall(r"[^\W\d_]+",text.lower())

print(f"{len(woerter)} Wörter insgesamt")
print(f"{len(set(woerter))} verschiedene Wörter")
print()
print(woerter[:30])

## Die häufigsten Wörter

`Counter` zählt, wie oft jedes Wort vorkommt. Die Funktion `zeichne_haeufigste` zeichnet die häufigsten als Balkendiagramm, wir brauchen sie gleich noch einmal.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt


def zeichne_haeufigste(woerter, titel, anzahl=15):
    haeufigste = Counter(woerter).most_common(anzahl)
    namen = [wort for wort, _ in haeufigste]
    werte = [zahl for _, zahl in haeufigste]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(namen[::-1], werte[::-1])
    ax.set_xlabel("Häufigkeit")
    ax.set_title(titel)
    plt.show()


zeichne_haeufigste(woerter, "Häufigste Wörter in Hänsel und Gretel")

Ernüchternd: "und", "die", "der", "sie". Die häufigsten Wörter sind in jedem Text dieselben und sagen nichts über den Inhalt.

## Stoppwörter entfernen

Solche Füllwörter heißen **Stoppwörter**. Wir legen eine Liste an und behalten nur Wörter, die nicht darin stehen.

**Aufgabe:** Schau auf das neue Diagramm. Steht dort noch ein Wort, das dich nicht interessiert? Füge es der Liste hinzu und führe die Zelle neu aus.

In [ ]:
stoppwoerter = {
    "und", "die", "der", "das", "sie", "er", "es", "in", "zu", "den", "dem", "des",
    "nicht", "ein", "eine", "einen", "einem", "einer", "ich", "du", "wir", "ihr",
    "ihm", "ihn", "mir", "mich", "dir", "dich", "sich", "da", "so", "war", "waren",
    "aber", "mit", "auf", "als", "von", "an", "ist", "auch", "wie", "nach", "daß",
    "dass", "im", "hatte", "hatten", "dann", "noch", "was", "nun", "sein", "ihre",
    "wenn", "aus", "bei", "doch", "denn", "nur", "um", "über", "vor", "hat", "wieder",
    "schon", "alle", "ganz", "wollte", "will", "wo", "ja", "nichts", "mehr", "gar",
}

gefiltert = [wort for wort in woerter if wort not in stoppwoerter]

zeichne_haeufigste(gefiltert, "Häufigste Wörter in Hänsel und Gretel, ohne Stoppwörter")

## Zwei Märchen vergleichen

Was unterscheidet ein Märchen vom anderen? Die Funktion `lade_woerter` fasst die bisherigen Schritte zusammen: einlesen, zerlegen, Stoppwörter entfernen. Dann suchen wir Wörter, die in einem Märchen häufig sind und im anderen gar nicht vorkommen.

**Aufgabe:** Vergleiche zwei andere Märchen.

In [ ]:
def lade_woerter(dateiname):
    text = (ordner / dateiname).read_text(encoding="utf-8")
    woerter = re.findall(r"[^\W\d_]+",text.lower())
    return [wort for wort in woerter if wort not in stoppwoerter]


a = lade_woerter("15_haensel_und_gretel.txt")
b = lade_woerter("12_rapunzel.txt")

nur_in_a = Counter(wort for wort in a if wort not in set(b))
nur_in_b = Counter(wort for wort in b if wort not in set(a))

print("Nur in Hänsel und Gretel:", nur_in_a.most_common(10))
print()
print("Nur in Rapunzel:        ", nur_in_b.most_common(10))

## Bonus: Eine Wordcloud

Für Wordclouds gibt es das Paket `wordcloud`. Es ist auf dem Hub nicht vorinstalliert, aber das ist kein Problem: Mit `%pip install` installierst du Pakete selbst in deinen eigenen Server. Das dauert einen Moment.

In [ ]:
%pip install wordcloud

Jetzt steht das Paket zur Verfügung. `generate_from_frequencies` nimmt unsere gezählten Wörter und setzt sie nach Häufigkeit skaliert zusammen.

In [ ]:
from wordcloud import WordCloud

wolke = WordCloud(width=900, height=450, background_color="white")
wolke.generate_from_frequencies(Counter(gefiltert))

fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(wolke)
ax.axis("off")
plt.show()

## Was das zeigt

Textdateien, Zählen, Diagramme und ein selbst nachinstalliertes Paket, alles in einem Notebook. Wer die Analyse weitergibt, gibt die Texte und die Stoppwortliste gleich mit. Jede Entscheidung ist nachvollziehbar und mit **Restart Kernel and Run All Cells** wiederholbar.